In [9]:
from spacerocks.nbody import Simulation, Integrator, Force
from spacerocks.time import Time
from spacerocks.spice import SpiceKernel
from spacerocks import SpaceRock



In [10]:
kernel = SpiceKernel.defaults()


Using default configuration:
  Kernel paths: ["/Users/thomasruch/.spacerocks/spice"]
  Download directory: "/Users/thomasruch/.spacerocks/spice"
  Auto-download: true

Processing kernel: latest_leapseconds.tls
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls
Loading kernel: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls

Processing kernel: de440s.bsp
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/de440s.bsp
Loading kernel: /Users/thomasruch/.spacerocks/spice/de440s.bsp

Processing kernel: earth_1962_240827_2124_combined.bpc
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc
Loading kernel: /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc

Processing kernel: codes_300ast_20100725.bsp
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.bsp
Loading kernel: /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.bsp


In [3]:
kernel

SpiceKernel:
  - /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls
  - /Users/thomasruch/.spacerocks/spice/de440s.bsp
  - /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc
  - /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.bsp
  - /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.tf

In [18]:
# Create a simulation with the giant planets
epoch = Time.now()
sim = Simulation.giants(epoch=epoch, reference_plane="ECLIPJ2000", origin="SSB")

# Add relativistic effects
sim.add_force(Force.solar_gr())
sim.add_force(Force.solar_j2())

# Configure integration
sim.set_integrator(Integrator.ias15(timestep=1.0))  # 1-day timestep
sim.move_to_center_of_mass()

# Integrate forward 1 year
future_epoch = epoch + 365.25
sim.integrate(future_epoch)

# Get final positions
jupiter = sim.get_particle("jupiter barycenter")
saturn = sim.get_particle("saturn barycenter")
print(f"Jupiter position: {jupiter.position}")
print(f"Saturn position: {saturn.position}")

Jupiter position: (-1.8481886625435187, 4.876897193602802, 0.021132464797636318)
Saturn position: (9.493932925459712, 0.36919218835859585, -0.38442713519963534)


In [22]:
# Start with empty simulation
sim = Simulation()
epoch = Time.now()

print(epoch)

# Add Sun and a custom body
sun = SpaceRock.from_spice("sun", epoch=epoch, reference_plane="ECLIPJ2000", origin="SSB")
sim.add(sun)

# Add custom body on eccentric orbit
body = SpaceRock.from_kepler(
    name="custom_body",
    q=2.5,       # perihelion distance [AU]
    e=0.85,      # eccentricity
    inc=0.4,     # inclination [rad]
    node=1.2,    # longitude of ascending node [rad]
    arg=2.1,     # argument of perihelion [rad]
    true_anomaly=0.0,  # true anomaly [rad]
    epoch=epoch,
    reference_plane="ECLIPJ2000",
    origin="SSB"
)
body.set_mass(1e-10)  # Set mass in solar masses
sim.add(body)

# Move to center of mass and integrate
sim.move_to_center_of_mass()
sim.integrate(epoch + 100.0)  # Integrate for 100 days

Time: 2460697.2771643517 UTC JD


In [23]:
sim.get_particle(body.name)

SpaceRock: custom_body
position: [[-1.9230760017109423, -1.8506746966685648, -0.27745451535599974]]
velocity: [[0.0055958804609810954, -0.009755876174290613, -0.008712020275687881]]
epoch: 2460797.2771643517 UTC JD
reference_plane: J2000
origin: simulation_barycenter

In [38]:
from spacerocks import RockCollection

In [39]:
epoch = Time.now()
collection = RockCollection()


In [40]:
Jupiter = SpaceRock.from_horizons("Jupiter Barycenter", epoch)
Neptune = SpaceRock.from_horizons("Neptune Barycenter", epoch)
Mars = SpaceRock.from_horizons("Mars Barycenter", epoch)

In [41]:
collection.add(Jupiter)
collection.add(Neptune)
collection.add(Mars)

In [42]:
collection.a()

array([ 5.17713205, 30.06945221,  1.51794713])

In [44]:
under_10_collection = collection.filter(collection.a() < 10)

In [45]:
under_10_collection

RockCollection: 2 rocks

In [47]:
for rock in under_10_collection:
    print(rock.a())

5.177132054094702
1.517947133743725


In [51]:
mask = [rock.e() > 0.05 for rock in collection]
filtered_collection = collection.filter(mask)

In [54]:
filtered_collection[0].e()

0.09827502028901582

In [1]:
from spacerocks.spice import SpiceKernel
from spacerocks import SpaceRock
from spacerocks.time import Time

In [2]:
# Manually instantiate a SpiceKernel object, and load relevant spice kernels located on your local machine 
kernel = SpiceKernel()
kernel.load("/Users/thomasruch/Gerdes/leap_seconds.tls")
kernel.load("/Users/thomasruch/Gerdes/de440s.bsp")

Loading kernel: /Users/thomasruch/Gerdes/leap_seconds.tls
Loading kernel: /Users/thomasruch/Gerdes/de440s.bsp


In [3]:
kernel

SpiceKernel:
  - /Users/thomasruch/Gerdes/leap_seconds.tls
  - /Users/thomasruch/Gerdes/de440s.bsp

In [4]:
kernel1 = SpiceKernel.from_config("/Users/thomasruch/srnew1/spacerocks/config.toml")

Loading configuration from /Users/thomasruch/srnew1/spacerocks/config.toml

Configuration loaded:
  Kernel paths: ["/Users/thomasruch/Gerdes", "/Users/thomasruch/.spacerocks/spice/"]
  Download directory: "/Users/thomasruch/.spacerocks/spice"
  Auto-download: true

Processing kernel: latest_leapseconds.tls
➜ Downloading kernel...
    Saving to /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls
    Download complete
Loading kernel: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls

Processing kernel: de440s.bsp
✓ Found existing kernel at: /Users/thomasruch/Gerdes/de440s.bsp
Loading kernel: /Users/thomasruch/Gerdes/de440s.bsp

Processing kernel: earth_1962_240827_2124_combined.bpc
✓ Found existing kernel at: /Users/thomasruch/Gerdes/earth_1962_240827_2124_combined.bpc
Loading kernel: /Users/thomasruch/Gerdes/earth_1962_240827_2124_combined.bpc


In [5]:
kernel1 

SpiceKernel:
  - /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls
  - /Users/thomasruch/Gerdes/de440s.bsp
  - /Users/thomasruch/Gerdes/earth_1962_240827_2124_combined.bpc

In [6]:
kernel2 = SpiceKernel.defaults(download=True)


Using default configuration:
  Kernel paths: ["/Users/thomasruch/.spacerocks/spice"]
  Download directory: "/Users/thomasruch/.spacerocks/spice"
  Auto-download: true

Processing kernel: latest_leapseconds.tls
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls
Loading kernel: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls

Processing kernel: de440s.bsp
➜ Downloading kernel...
    Saving to /Users/thomasruch/.spacerocks/spice/de440s.bsp
    Download complete
Loading kernel: /Users/thomasruch/.spacerocks/spice/de440s.bsp

Processing kernel: earth_1962_240827_2124_combined.bpc
➜ Downloading kernel...
    Saving to /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc
    Download complete
Loading kernel: /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc


In [1]:
from spacerocks.time import Time
from spacerocks.observing import Observatory
from spacerocks.time import Time
from spacerocks import SpaceRock
from spacerocks.spice import SpiceKernel

In [2]:
kernel = SpiceKernel()
kernel.load("/Users/thomasruch/Gerdes/leap_seconds.tls")
kernel.load("/Users/thomasruch/Gerdes/de440s.bsp")
kernel.load("/Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc")

Loading kernel: /Users/thomasruch/Gerdes/leap_seconds.tls
Loading kernel: /Users/thomasruch/Gerdes/de440s.bsp
Loading kernel: /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc


In [3]:
# Create observatory and time
telescope = Observatory.from_obscode('F51')
epoch = Time.now()

# Create and configure target
target = SpaceRock.from_horizons(
    "Apophis", 
    epoch,
    reference_plane="J2000",
    origin='SSB'
)
target.set_absolute_magnitude(19.7)

# Get observation
observer = telescope.at(epoch)
obs = target.observe(observer)

print(f"RA: {obs.ra}")
print(f"Dec: {obs.dec}")
print(f"Range: {obs.range} AU")
print(f"Magnitude: {obs.mag}")

RA: 5.260928972940125
Dec: -0.3281633047446638
Range: 1.7486690602683705 AU
Magnitude: 20.90439792814383


In [4]:
# Set up observatories
mauna_kea = Observatory.from_obscode('568')
siding_spring = Observatory.from_obscode('413')

In [5]:
# Create time series
step = 100
n_points = 10
start_jd = Time.now().jd()
epochs = [Time((start_jd + i*step), 'UTC', 'JD') for i in range(n_points)]
observations = []

In [7]:
for epoch in epochs:
    # Observe from both sites
    target.analytic_at(epoch)
    obs_mk = target.observe(mauna_kea.at(epoch))
    obs_sso = target.observe(siding_spring.at(epoch))
    observations.append((obs_mk, obs_sso))

AttributeError: 'builtins.SpaceRock' object has no attribute 'analytic_at'

In [8]:
# Create SpaceRock
epoch = Time.now()
arrokoth = SpaceRock.from_horizons("Arrokoth", epoch)

# Calculate orbital elements
print(f"Semi-major axis: {arrokoth.a()} AU")
print(f"Eccentricity: {arrokoth.e()}")

# Set physical properties
arrokoth.set_mass(1e-10)
arrokoth.set_absolute_magnitude(15.0)

Semi-major axis: 44.23095719620317 AU
Eccentricity: 0.037802504828944004


In [13]:
from spacerocks.observing import Observer, Observatory, Observation
# Set up objects
epoch = Time.now()
asteroid = SpaceRock.from_horizons("Ceres", epoch, "J2000")
telescope = Observatory.from_obscode("w84")
observer = telescope.at(epoch)  

# Calculate ephemeris
observation = asteroid.observe(observer)
print(f"RA: {observation.ra} rad")
print(f"Dec: {observation.dec} rad")

RA: 5.616024589149868 rad
Dec: -0.39244776678263826 rad


In [21]:
time = Time(2905840.9, "utc", "jd")

In [22]:
time.to_tdb()

Time: 2905840.9008007217 TDB JD

In [23]:
time

Time: 2905840.9008007217 TDB JD

In [26]:
time_in_utc = time.utc()

In [27]:
time

Time: 2905840.9008007217 TDB JD

In [28]:
time_in_utc

Time: 2905840.9 UTC JD

In [30]:
time.mjd()

505840.4008007217

In [32]:
time.calendar()

'27 Oct 3243'

In [33]:
time.iso()

'3243-10-27T09:35:59.000Z'

In [34]:
time

Time: 2905840.9008007217 TDB JD

In [35]:
time + 1000

Time: 2906840.9008007217 TDB JD

In [38]:
time_new = Time(20424.5, 'tdx', 'mjd')

ValueError: Invalid timescale: 'tdx'. Did you mean 'tdb'?. Needs to be 'utc', 'tdb', 'tt', or 'tai'.

In [37]:
time_new

(20424.5, 'tdx', 'mjd')

In [7]:
from spacerocks import MPCHandler

In [8]:
handler = MPCHandler()

In [47]:
# Create a dataframe of all the Near-Earth Asteroids (NEAs) from the MPC data files
nea_df = handler.fetch_data(
    catalog="mpcorb_extended", # https://minorplanetcenter.net/data details available catalogs
    orbit_type= "Hilda",  # None means get all types, and is the default
    output_format="dataframe" # Options: "dataframe", "rocks": Default is "dataframe", "rocks" returns a RockCollection 
)

Using existing json file: /Users/thomasruch/.spacerocks/mpc/mpcorb_extended.json.gz


In [48]:
nea_df

,H,G,Epoch,M,Peri,Node,i,e,a,Principal_desig,orbit_type
0,7.75,0.15,2460600.5,63.64996,39.32434,228.08798,7.82939,0.138727,3.970658,A875 VC,Hilda
1,7.91,0.15,2460600.5,146.33147,271.30376,175.42328,6.17472,0.168317,3.994557,A878 SA,Hilda
2,7.64,0.15,2460600.5,114.77809,169.05178,129.80984,4.63717,0.025093,3.920176,A892 QB,Hilda
3,8.39,0.15,2460600.5,345.48552,67.12248,18.74267,12.58515,0.211862,3.985101,A893 EF,Hilda
4,9.62,0.15,2460600.5,157.46959,175.11255,256.17038,2.09258,0.217457,4.014617,A902 YE,Hilda
...,...,...,...,...,...,...,...,...,...,...,...
6357,17.40,0.15,2458340.5,330.77136,161.57387,206.89741,3.32396,0.189466,3.897767,2018 PU25,Hilda
6358,18.50,0.15,2458340.5,338.79285,219.38379,146.60116,6.88106,0.231140,3.886307,2018 PQ30,Hilda
6359,17.10,0.15,2458760.5,350.95844,251.46347,148.73943,1.41872,0.174703,3.949616,2019 SN122,Hilda
6360,16.90,0.15,2458780.5,339.24413,275.67108,142.23657,3.78468,0.155806,3.924514,2019 US68,Hilda


In [51]:
from spacerocks.orbfit import gauss
from spacerocks.observing import Observer
from spacerocks.time import Time

In [54]:
# Create or load three observations
time1 = Time.now()
time2 = time1 + 150
time3 = time2 + 150

observer1 = Observatory.from_obscode('w84').at(time1)
observer2 = Observatory.from_obscode('w84').at(time2)
observer3 = Observatory.from_obscode('w84').at(time3)

obs1 = Observation.from_astrometry(epoch, 3.45, 0.24, observer1)
obs2 = Observation.from_astrometry(epoch, , dec, observer2)
obs3 = Observation.from_astrometry(epoch, ra, dec, observer3)


NameError: name 'ra' is not defined

In [1]:
from spacerocks.nbody import Simulation, Integrator, Force
from spacerocks.time import Time
from spacerocks.spice import SpiceKernel
from spacerocks import SpaceRock


kernel = SpiceKernel.defaults()


Using default configuration:


  Kernel paths: ["/Users/thomasruch/.spacerocks/spice"]
  Download directory: "/Users/thomasruch/.spacerocks/spice"
  Auto-download: true

Processing kernel: latest_leapseconds.tls
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls
Loading kernel: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls

Processing kernel: de440s.bsp
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/de440s.bsp
Loading kernel: /Users/thomasruch/.spacerocks/spice/de440s.bsp

Processing kernel: earth_1962_240827_2124_combined.bpc
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc
Loading kernel: /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc

Processing kernel: codes_300ast_20100725.bsp
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.bsp
Loading kernel: /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.bsp

Processing kernel: codes_300a

In [2]:
from spacerocks import MPCHandler
import pandas as pd

In [3]:
handler = MPCHandler()

In [4]:
objs, response = handler.get_detections("davidgerdes", formats=["ADES_DF", "OBS_DF", "XML"])
#obs, response = handler.get_detections("2021 CS1")
# if response:
#     ades_df = pd.DataFrame(response[0]['ADES_DF'])
#     obs_df = pd.DataFrame(response[0]['OBS_DF'])
#     xml_string = response[0]['XML']


In [5]:
response

[{'ADES_DF': [{'Obstype': 'optical',
    'artsat': None,
    'astcat': 'UNK',
    'band': 'R',
    'com': None,
    'ctr': None,
    'dec': '13.48519',
    'decstar': None,
    'delay': None,
    'deltadec': None,
    'deltara': None,
    'deprecated': None,
    'disc': '*',
    'dist': None,
    'doppler': None,
    'exp': None,
    'frq': None,
    'localuse': None,
    'logsnr': None,
    'mag': '19.4',
    'mode': 'CCD',
    'notes': None,
    'nstars': None,
    'nucmag': None,
    'obscenter': None,
    'obsid': 'JNC9uv000000DOOU0100000Ut',
    'obstime': '2000-02-06T07:59:45.024Z',
    'pa': None,
    'permid': '208117',
    'photap': None,
    'photcat': None,
    'pos1': None,
    'pos2': None,
    'pos3': None,
    'poscov11': None,
    'poscov12': None,
    'poscov13': None,
    'poscov22': None,
    'poscov23': None,
    'poscov33': None,
    'precdec': '0.1',
    'precra': '0.01',
    'prectime': 10.0,
    'prog': '01',
    'provid': '2000 CJ119',
    'ra': '150.08196',
  

In [6]:
len(objs)

415

In [7]:
objs[100]


Observation:
(RA=111.8725°, Dec=28.5482°)
mag = 20.0
epoch: 2456307.7756481483 UTC JD

In [16]:
ades_df

,Obstype,artsat,astcat,band,com,ctr,dec,decstar,delay,deltadec,...,rmstime,seeing,stn,subfmt,subfrm,sys,trkid,trksub,trx,unctime
0,optical,None,UNK,R,None,None,13.48519,None,None,None,...,None,None,695,M92,None,None,000003o-bf,MB5190,None,None
1,optical,None,UNK,None,None,None,13.49061,None,None,None,...,None,None,695,M92,None,None,000003o-bf,MB5190,None,None
2,optical,None,USNOSA1,V,None,None,13.80372,None,None,None,...,None,None,691,M92,None,None,000003G-LH,A2E41j,None,None
3,optical,None,USNOSA1,V,None,None,13.80467,None,None,None,...,None,None,691,M92,None,None,000003G-LH,A2E41j,None,None
4,optical,None,USNOSA1,V,None,None,13.80550,None,None,None,...,None,None,691,M92,None,None,000003G-LH,A2E41j,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410,optical,None,Gaia3E,Pw,None,None,-12.269759,None,None,None,...,None,None,F51,A17,None,None,00000IHUaX,P1201GQ,None,None
411,optical,None,Gaia2,G,None,None,-6.138830,None,None,None,...,None,None,V00,A17,None,None,00000IV6GU,C0Z6PW5,None,None
412,optical,None,Gaia2,G,None,None,-6.138260,None,None,None,...,None,None,V00,A17,None,None,00000IV6GU,C0Z6PW5,None,None
413,optical,None,Gaia2,G,None,None,-6.137480,None,None,None,...,None,None,V00,A17,None,None,00000IV6GU,C0Z6PW5,None,None


In [15]:
ades_df['obstime'][0]

'2000-02-06T07:59:45.024Z'

In [10]:
from spacerocks.time import Time

In [11]:
isot_time = Time.from_isot(ades_df['obstime'][0])

In [12]:
isot_time

Time: 2451580.8331597224 UTC JD

In [37]:
ades_df.head()

,Obstype,artsat,astcat,band,com,ctr,dec,decstar,delay,deltadec,...,rmstime,seeing,stn,subfmt,subfrm,sys,trkid,trksub,trx,unctime
0,optical,None,UNK,R,None,None,13.48519,None,None,None,...,None,None,695,M92,None,None,000003o-bf,MB5190,None,None
1,optical,None,UNK,None,None,None,13.49061,None,None,None,...,None,None,695,M92,None,None,000003o-bf,MB5190,None,None
2,optical,None,USNOSA1,V,None,None,13.80372,None,None,None,...,None,None,691,M92,None,None,000003G-LH,A2E41j,None,None
3,optical,None,USNOSA1,V,None,None,13.80467,None,None,None,...,None,None,691,M92,None,None,000003G-LH,A2E41j,None,None
4,optical,None,USNOSA1,V,None,None,13.80550,None,None,None,...,None,None,691,M92,None,None,000003G-LH,A2E41j,None,None


In [39]:
print(xml_string)

<?xml version="1.0" encoding="UTF-8"?>
<ades version="2017">
  <optical>
    <permID>208117</permID>
    <provID>2000 CJ119</provID>
    <trkSub>MB5190</trkSub>
    <obsID>JNC9uv000000DOOU0100000Ut</obsID>
    <trkID>000003o-bf</trkID>
    <mode>CCD</mode>
    <stn>695</stn>
    <prog>01</prog>
    <obsTime>2000-02-06T07:59:45.024Z</obsTime>
    <ra>150.08196</ra>
    <dec>13.48519</dec>
    <astCat>UNK</astCat>
    <mag>19.4</mag>
    <band>R</band>
    <ref>MPS    16858</ref>
    <disc>*</disc>
    <subFmt>M92</subFmt>
    <precTime>10</precTime>
    <precRA>0.01</precRA>
    <precDec>0.1</precDec>
  </optical>
  <optical>
    <permID>208117</permID>
    <provID>2000 CJ119</provID>
    <trkSub>MB5190</trkSub>
    <obsID>JNC9uv000000DOOU0100000Uu</obsID>
    <trkID>000003o-bf</trkID>
    <mode>CCD</mode>
    <stn>695</stn>
    <prog>01</prog>
    <obsTime>2000-02-06T11:20:49.920Z</obsTime>
    <ra>150.04867</ra>
    <dec>13.49061</dec>
    <astCat>UNK</astCat>
    <ref>MPS    16858</r

In [62]:
import pandas as pd
a, b = handler.get_detections("~0ZTC", formats=["ADES_DF", "OBS_DF", "XML"])

In [63]:
dff = pd.DataFrame(a['ADES_DF'])

In [64]:
dff['permid']

0      756350
1      756350
2      756350
3      756350
4      756350
        ...  
123    756350
124    756350
125    756350
126    756350
127    756350
Name: permid, Length: 128, dtype: object

In [65]:
dff['provid']

0      2019 BD8
1      2019 BD8
2      2019 BD8
3      2019 BD8
4      2019 BD8
         ...   
123    2019 BD8
124    2019 BD8
125    2019 BD8
126    2019 BD8
127    2019 BD8
Name: provid, Length: 128, dtype: object

In [1]:
from spacerocks import SpaceRock
from spacerocks.time import Time
from spacerocks.nbody import Simulation, Integrator, Force
from spacerocks.spice import SpiceKernel

epoch = Time.now()
rock = SpaceRock.from_horizons("Ceres", epoch)

In [2]:
kernel = SpiceKernel.defaults()


Using default configuration:
  Kernel paths: ["/Users/thomasruch/.spacerocks/spice"]
  Download directory: "/Users/thomasruch/.spacerocks/spice"
  Auto-download: true

Processing kernel: latest_leapseconds.tls
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls
Loading kernel: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls

Processing kernel: de440s.bsp
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/de440s.bsp
Loading kernel: /Users/thomasruch/.spacerocks/spice/de440s.bsp

Processing kernel: earth_1962_240827_2124_combined.bpc
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc
Loading kernel: /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc

Processing kernel: codes_300ast_20100725.bsp
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.bsp
Loading kernel: /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.bsp


In [3]:
kernel

SpiceKernel:
  - /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls
  - /Users/thomasruch/.spacerocks/spice/de440s.bsp
  - /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc
  - /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.bsp
  - /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.tf

In [4]:
epoch

Time: 2460710.342604167 UTC JD

In [5]:
in_1000_days = epoch + 100

In [7]:
in_1000_days

Time: 2460810.290335648 UTC JD

In [8]:
rock.a(), rock.e(), rock.inc(), rock.node(), rock.arg(), rock.true_anomaly()

(2.7613359699553577,
 0.08042484063349035,
 0.18493395996895648,
 1.3994907052357999,
 1.2560696659059833,
 3.0043362707873715)

In [9]:
sim = Simulation.horizons(epoch, "ECLIPJ2000", "Sun")

In [10]:
rock.position, rock.velocity

((2.375376470743992, -1.732219432634831, -0.4931013448692911),
 (0.005603875158696297, 0.007711784039114464, -0.0007870530236765594))

In [11]:
#rock_prop = rock.analytic_at(epoch + 1000)
rock.analytic_propagate(in_1000_days)

In [12]:
rock.a(), rock.e(), rock.inc(), rock.node(), rock.arg(), rock.true_anomaly()

(2.7613359699557707,
 0.08042484063336781,
 0.18493395996895648,
 1.3994907052357999,
 1.2560696659091493,
 3.3249190683408285)

In [13]:
rock.position, rock.velocity

((2.793979999452294, -0.8794823534834006, -0.5430702354041189),
 (0.002689101678200921, 0.009184215468348807, -0.00020280744241320476))

In [1]:
from spacerocks import MPCHandler
from spacerocks.time import Time
from spacerocks.nbody import Simulation, Force, Integrator
from spacerocks.spice import SpiceKernel
from spacerocks import SpaceRock
from spacerocks.transforms import calc_true_anomaly_from_mean_anomaly
from spacerocks import RockCollection

import numpy as np

In [2]:
kernel = SpiceKernel.defaults()


Using default configuration:
  Kernel paths: ["/Users/thomasruch/.spacerocks/spice"]
  Download directory: "/Users/thomasruch/.spacerocks/spice"
  Auto-download: true

Processing kernel: latest_leapseconds.tls
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls
Loading kernel: /Users/thomasruch/.spacerocks/spice/latest_leapseconds.tls

Processing kernel: de440s.bsp
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/de440s.bsp
Loading kernel: /Users/thomasruch/.spacerocks/spice/de440s.bsp

Processing kernel: earth_1962_240827_2124_combined.bpc
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc
Loading kernel: /Users/thomasruch/.spacerocks/spice/earth_1962_240827_2124_combined.bpc

Processing kernel: codes_300ast_20100725.bsp
✓ Found existing kernel at: /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.bsp
Loading kernel: /Users/thomasruch/.spacerocks/spice/codes_300ast_20100725.bsp


In [3]:
# Create a rockcollection with the MPC data
handler = MPCHandler()
nea_collection = handler.fetch_data(
    catalog = "nea_extended",
    output_format = "dataframe"
)

print(f"Loaded {len(nea_collection)} NEAs")


Using existing json file: /Users/thomasruch/.spacerocks/mpc/nea_extended.json.gz
Loaded 37398 NEAs


In [4]:
nea_collection

,H,G,Epoch,M,Peri,Node,i,e,a,Principal_desig,orbit_type
0,10.41,0.15,2460600.5,86.66754,178.91030,304.27434,10.82773,0.222691,1.458181,A898 PA,Amor
1,15.59,0.15,2460600.5,148.45068,156.21553,183.85715,11.57526,0.546779,2.636157,A911 TB,Amor
2,13.79,0.15,2460600.5,340.19843,350.47424,110.42302,9.39880,0.571093,2.472525,A918 AA,Amor
3,9.18,0.15,2460600.5,6.98496,132.49616,215.49497,26.68673,0.532826,2.665299,A924 UB,Amor
4,17.37,0.15,2460600.5,271.70081,26.71763,171.26079,11.86849,0.434718,1.920154,1932 EA1,Amor
...,...,...,...,...,...,...,...,...,...,...,...
37393,19.88,0.15,2460600.5,16.79916,142.73350,155.01613,6.40102,0.397912,2.136127,2021 NF1,Amor
37394,18.43,0.15,2460600.5,261.16611,336.66141,30.64476,10.57609,0.597588,2.632801,2021 NS5,Amor
37395,18.93,0.15,2460600.5,220.33161,344.71167,265.00727,25.96643,0.511470,1.686187,2023 PB,Apollo
37396,18.96,0.15,2460600.5,118.61192,233.96853,153.80370,38.47990,0.368584,1.961853,2023 QN,Amor


In [5]:
collection = RockCollection()
# Add all elements in nea_collection to rock_collection one row at a time
for i, row in nea_collection.iterrows():
    # Create a rock object
    rock = SpaceRock.from_kepler(name = row["Principal_desig"],
                                 q = row["a"]* (1 - row["e"]),
                                 e = row["e"],
                                 inc = np.radians(row["i"]),
                                 arg = np.radians(row["Peri"]),
                                 node = np.radians(row["Node"]),
                                 true_anomaly = (calc_true_anomaly_from_mean_anomaly(row["e"], np.radians(row["M"]))),
                                 epoch = Time(row["Epoch"], "utc", "jd"),
                                 reference_plane = "ECLIPJ2000",
                                 origin = "SSB")
    # Add the rock object to the rock_collection
    collection.add(rock)

In [6]:
collection_from_method = RockCollection.from_mpc(
    catalog = "nea_extended"
)

Using existing json file: /Users/thomasruch/.spacerocks/mpc/nea_extended.json.gz


In [9]:
collection_from_method[100]

SpaceRock: 1993 EA
position: [[-1.9783643542677147, -0.12022086321164346, 0.1751284828682502]]
velocity: [[0.0019190491681180567, -0.007805864767178648, -8.466564880793617e-5]]
epoch: 2460600.5 UTC JD
reference_plane: ECLIPJ2000
origin: SSB

In [10]:
collection[100]

SpaceRock: 1993 EA
position: [[-1.9783643542677147, -0.12022086321164346, 0.1751284828682502]]
velocity: [[0.0019190491681180567, -0.007805864767178648, -8.466564880793617e-5]]
epoch: 2460600.5 UTC JD
reference_plane: ECLIPJ2000
origin: SSB